# Parameter Files

So far every model has been assembled by hand in Python. That is great for exploration, but science workflows benefit from reproducibility: being able to re-run or share an exact configuration without distributing a notebook. TauREx `.par` files are the answer — a simple INI-style format where each section corresponds to one Python component.

The mapping is direct and transparent:
- `[Temperature]` → temperature profile
- `[Pressure]` → pressure grid
- `[Chemistry]` → gas mixture
- `[Model]` → forward-model geometry and contributions

The same model we built manually in [Transmission spectrum basics](https://escience-taurex.github.io/taurex3/examples/03_transmission_basics.html), [Emission spectrum basics](https://escience-taurex.github.io/taurex3/examples/04_emission_basics.html), and [Cloud models](https://escience-taurex.github.io/taurex3/examples/05_clouds.html) can be expressed in a dozen lines of text. More information about the full parameter-file syntax is [here](../user/taurex/inputfile.rst), and global settings (data paths, log level) are [here](../user/taurex/global.rst).

## Data Note

This notebook uses the opacity and CIA files set up in [Setup and opacity data](https://escience-taurex.github.io/taurex3/examples/01_setup_and_data.html). TauREx provides the software to work with these datasets; the files themselves are third-party products from [ExoMol](https://www.exomol.com) and [HITRAN](https://hitran.org/).

In [1]:
from pathlib import Path
import os
import sys

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / '_shared.py').exists():
        sys.path.insert(0, str(candidate))
        break
    nested = candidate / 'examples' / 'notebooks'
    if (nested / '_shared.py').exists():
        sys.path.insert(0, str(nested))
        break

from _shared import CIA_DIR, TMP_DIR, XSEC_DIR, ensure_opacity_data

ensure_opacity_data(download=False)
output_path = Path(os.path.relpath(TMP_DIR / 'input.par', Path.cwd()))
xsec_path = os.path.relpath(XSEC_DIR, Path.cwd())
cia_path = os.path.relpath(CIA_DIR, Path.cwd())

print('Opacity data located and ready.')
print(f'Parameter file will be written to: {output_path}')

Opacity data located and ready.
Parameter file will be written to: ../../examples/tmp/input.par


In [2]:
input_par = f'''[Global]
xsec_path = {xsec_path}
cia_path = {cia_path}

[Chemistry]
chemistry_type = taurex
fill_gases = H2,He
ratio = 0.1741

    [[H2O]]
    gas_type = constant
    mix_ratio = 1e-3

[Temperature]
profile_type = isothermal
T = 2000

[Pressure]
profile_type = Simple
atm_min_pressure = 1e-5
atm_max_pressure = 1e5
nlayers = 100

[Planet]
planet_type = Simple
planet_mass = 0.74
planet_radius = 1.38

[Star]
star_type = blackbody
temperature = 6117
radius = 1.16

[Model]
model_type = transmission

    [[Absorption]]
'''

output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(input_par)
print(input_par)

[Global]
xsec_path = ../../examples/tmp/xsec
cia_path = ../../examples/tmp/cia

[Chemistry]
chemistry_type = taurex
fill_gases = H2,He
ratio = 0.1741

    [[H2O]]
    gas_type = constant
    mix_ratio = 1e-3

[Temperature]
profile_type = isothermal
T = 2000

[Pressure]
profile_type = Simple
atm_min_pressure = 1e-5
atm_max_pressure = 1e5
nlayers = 100

[Planet]
planet_type = Simple
planet_mass = 0.74
planet_radius = 1.38

[Star]
star_type = blackbody
temperature = 6117
radius = 1.16

[Model]
model_type = transmission

    [[Absorption]]



## Loading the Parameter File

`ParameterParser` reads a `.par` file and reconstructs the full model hierarchy — exactly the same objects that would be created by hand in Python. This round-trip confirms that the declarative and programmatic approaches are equivalent.

Full syntax details, dynamic-header behaviour, and required section names are documented [here](../user/taurex/inputfile.rst).

### Parameter files for multimodel runs

Parameter files are also the entry point for TauREx composite forward models. Set `model_type = multi_transit`, `multi_eclipse`, or `multi_directimage` in the main file, then list the regional files in `parfiles`. Each regional file can carry its own temperature, chemistry, pressure, and contribution blocks, while `fractions` in the main file controls the area weighting.

In [3]:
from taurex.parameter import ParameterParser

parser = ParameterParser()
parser.read(str(output_path))
parser.setup_globals()
model = parser.generate_appropriate_model()

print(type(model).__name__)
print('Contributions:', [c.name for c in model.contribution_list])

TransmissionModel
Contributions: ['Absorption']


In [4]:
print('Temperature object:', type(model.temperature).__name__)
print('Chemistry object:', type(model.chemistry).__name__)
print('Pressure object:', type(model.pressure).__name__)

Temperature object: Isothermal
Chemistry object: TaurexChemistry
Pressure object: SimplePressureProfile
